<a href="https://colab.research.google.com/github/wwieder/ADF/blob/main/snotel_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plotting data from Snotel sites

This is a notebook that uses python to dowload and plot snowdepth observations from Snotel sites

First we need to install libraries to run our python notebook.
- Most of the libraries are 'standard' (pandas, matplotlib)
- We'll aso use a more specialized (metloom) to get Snotel data

----

To 'run' a cell, you can just hit the play button next to each code block.

Once the code has been executed successfully you'll see a number [1] and green check mark ✔ appear to the left of the code block.  Try it below

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
!pip install -q metloom
from metloom.pointdata import SnotelPointData


This is an example of a text block.

Below you'll see code block.  Comments in the code are helpful to understand what's being done and are on lines preceeded wtih a hash (#)

You can find information about sites and sample plots at this website
https://nwcc-apps.sc.egov.usda.gov/site-plots/?state=CO


-----
-----

# A) First we'll download Snotel data and make a quick plot of the dataset

In [ ]:
# 1. Define station and variables (e.g., Niwot, CO: 663)
# We'll use a function called SnotelPointData:
#    which requires the station identification and name

snotel_point = SnotelPointData("663:CO:SNTL", "Niwot")
variables = [snotel_point.ALLOWED_VARIABLES.SWE] # Snow Water Equivalent

In [ ]:
# 2. Fetch data, and store it in a dataframe we'll call (df)
df = snotel_point.get_daily_data(
    start_date=datetime(2000, 10, 1),
    end_date=datetime(2026, 5, 25),
    variables=variables
).reset_index()

In [ ]:
# 3. Now that you have your data. Let's look at how it's formatted.
df

### Look:
- How many rows are in your dataset?  
- What variables are included?
- What is SWE?

In [ ]:
# 4. Make a quick plot of the dataset
df.plot.line(x='datetime', y='SWE', title='SNOTEL SWE at Niwot')
plt.ylabel('Inches') ;

### Look:
- What do you notice about these data?
- What years had the the deepest snowpack?  
   - How much snow was there?
- What years had the thinnest snowpack?


----------
---------
# B) Transform data for a water year
Water managers use a calendar that's shifted for a "Water Year".  Water years typically start on Oct 1.

*Why would a water year be useful for thinking about snow accumulation and melt*

In [ ]:
# 1. First we'll convert our dataset to a pandas datafram
df = pd.DataFrame({"Date": df.datetime, "SWE_in": df.SWE.values})
df.set_index("Date", inplace=True)


In [ ]:
# 2. Create a function to define each water year
#    Calculate the Water Year integer (e.g., Nov 2023 belongs to WY 2024)
df['Water_Year'] = df.index.year + (df.index.month >= 10).astype(int)

# Calculate the continuous day number relative to Oct 1st
def get_day_of_water_year(dt):
    wy_start_year = dt.year if dt.month >= 10 else dt.year - 1
    # Create wy_start as a timezone-aware Timestamp, matching dt's timezone
    wy_start = pd.Timestamp(year=wy_start_year, month=10, day=1, tz=dt.tz)
    return (dt - wy_start).days + 1

df['WY_Day'] = df.index.map(get_day_of_water_year)


In [ ]:
# 3. Look at your new dataframe (df)
df

### Look:
- What variables are in this new dataframe?

In [ ]:
# 4. PLOTTING THE SNOTEL WATER YEAR GRAPH
#    This example is kind of complicated to explain (but feel free to ask questions)

plt.figure(figsize=(11, 7))

# Pivot data to align all water years by the same x-axis ('WY_Day')
for wy, group in df.groupby('Water_Year'):
    # Sort by day to prevent messy line drawing
    group = group.sort_values('WY_Day')

    # Optional: Highlight the latest/current water year dynamically
    if wy == df['Water_Year'].max():
        plt.plot(group['WY_Day'], group['SWE_in'],
                 label=f"WY {wy} (Current)", color='royalblue', linewidth=3)
    else:
      plt.plot(group['WY_Day'], group['SWE_in'],
               label=f"WY {wy}", alpha=0.6, linestyle='--')

plt.title='SNOTEL SWE at Niwot'
plt.xlabel('Day of water year')
plt.ylabel('Inches')
plt.legend() ;


### Look:
- How is this plot different than the previous one?
- What new information can you see from this kind of plot?
- What other information may be helpful to understanding the snowpack at this site?

*See if you can start using AI to help you make new plots.*